# Seconds to Minutes Cut My Model's Latency Effect by 99.7%

Companion notebook for the article. Run all cells to reproduce every number. Needs NumPy, pandas and scikit-learn.

From [*Math for ML Engineers: From Backprop to RLHF*](https://mikaelyemane.github.io/ml-math-code/) by Mikael Yemane.

In [1]:
%pip install -q numpy pandas scikit-learn

## Three leaks, measured

In [2]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(0)
latency_s = rng.gamma(2.0, 0.15, 2000)
retries = rng.poisson(1.0, 2000)
y = rng.random(2000) < 1 / (1 + np.exp(-(4.0 * latency_s + 0.8 * retries - 3.0)))

def effect_per_second(model, scale):
    if hasattr(model, "steps"):                       # pipeline: undo the scaler
        return model[-1].coef_[0, 0] / model[0].scale_[0] * scale
    return model.coef_[0, 0] * scale

for unit, scale in [("seconds", 1), ("milliseconds", 1000), ("minutes", 1 / 60)]:
    X = np.c_[latency_s * scale, retries]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.5, random_state=0)
    default = LogisticRegression().fit(Xtr, ytr)
    p = default.predict_proba(Xte)[:, 1]
    unpenalized = LogisticRegression(C=np.inf, max_iter=10_000).fit(Xtr, ytr)
    scaled = make_pipeline(StandardScaler(), LogisticRegression()).fit(Xtr, ytr)
    print(unit, round(effect_per_second(default, scale), 2), round(roc_auc_score(yte, p), 3),
          f"{np.mean(p > 0.5):.1%}", round(effect_per_second(unpenalized, scale), 2),
          round(effect_per_second(scaled, scale), 2))

seconds 3.7 0.785 17.4% 4.22 4.18
milliseconds 4.21 0.784 17.4% 4.22 4.18
minutes 0.01 0.76 8.1% 4.22 4.18


In [3]:
import pandas as pd

df = pd.DataFrame({"segment": ["a", "b", None, "a", None, "b"],
                   "revenue": [10, 20, 30, 40, 50, 60]})
print(df.revenue.sum())                                     # 210
print(df.groupby("segment").revenue.sum().sum())            # 130
print(df.groupby("segment", dropna=False).revenue.sum().sum())  # 210

210
130
210


In [4]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit

rng = np.random.default_rng(0)
t = np.arange(1500)
y = 0.05 * t + 10 * np.sin(2 * np.pi * t / 7) + rng.normal(0, 3, 1500)
X = np.c_[t, np.sin(2 * np.pi * t / 7), np.cos(2 * np.pi * t / 7)]
model = RandomForestRegressor(n_estimators=200, random_state=0)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0)  # shuffle=True by default
print(model.fit(Xtr, ytr).score(Xte, yte))                                  # R² 0.978
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, shuffle=False)   # train on past, test on future
print(model.fit(Xtr, ytr).score(Xte, yte))                                  # R² -0.303
print(cross_val_score(model, X, y))                                         # default, no shuffle: -0.17 0.58 0.53 0.70 -0.30
print(cross_val_score(model, X, y, cv=TimeSeriesSplit(5)))                  # -0.25 to 0.14 per fold

0.9778834275213526


-0.3029896598669557


[-0.16886805  0.57905792  0.53392008  0.7003187  -0.30298966]


[ 0.04364463  0.01071467 -0.25093688  0.01243642  0.14261144]


## In defense of the defaults

In [5]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, brier_score_loss

scores = {"default": [], "unpenalized": []}
wrong, sure_total = 0, 0
for seed in range(5):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(400, 50))                # 50 features, 200 training rows
    w = np.zeros(50); w[:5] = 1.5                 # only 5 features matter
    y = rng.random(400) < 1 / (1 + np.exp(-(X @ w)))
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.5, random_state=0)
    for name, m in [("default", LogisticRegression()),
                    ("unpenalized", LogisticRegression(C=np.inf, max_iter=10_000))]:
        q = m.fit(Xtr, ytr).predict_proba(Xte)[:, 1]
        sure = (q < 1e-6) | (q > 1 - 1e-6)        # near-certain predictions
        scores[name].append((roc_auc_score(yte, q), brier_score_loss(yte, q), sure.mean()))
        if name == "unpenalized":
            wrong += ((q[sure] > 0.5) != yte[sure]).sum(); sure_total += sure.sum()
for name, s in scores.items():
    print(name, np.mean(s, axis=0).round(3))     # AUC, Brier score, share near-certain
print("unpenalized near-certain calls that were wrong:", wrong, "of", sure_total)

default [0.879 0.154 0.002]
unpenalized [0.845 0.212 0.544]
unpenalized near-certain calls that were wrong: 90 of 544


## Seven rules for model code built on abstractions

In [6]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def make_model():
    return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, l1_ratio=0.0))

def test_predictions_do_not_depend_on_units():
    rng = np.random.default_rng(1)
    X = np.c_[rng.gamma(2.0, 0.15, 1000), rng.poisson(1.0, 1000)]
    y = rng.random(1000) < 1 / (1 + np.exp(-(4.0 * X[:, 0] + 0.8 * X[:, 1] - 3.0)))
    to_minutes = np.array([1 / 60, 1])
    p_seconds = make_model().fit(X, y).predict_proba(X)[:, 1]
    p_minutes = make_model().fit(X * to_minutes, y).predict_proba(X * to_minutes)[:, 1]
    assert np.allclose(p_seconds, p_minutes, atol=1e-6)

## Reproduce the numbers quoted in the text

In [7]:
test_predictions_do_not_depend_on_units()
print("invariance test passes for the scaled pipeline")
make_model = lambda: LogisticRegression()          # the bare model the article warns about
try:
    test_predictions_do_not_depend_on_units()
except AssertionError:
    print("and fails for a bare LogisticRegression()")
out = df.groupby("segment").agg(revenue=("revenue", "sum"), n=("revenue", "size"))
print("row-count check on the leak 2 data:", out["n"].sum(), "rows counted against", len(df))

invariance test passes for the scaled pipeline
and fails for a bare LogisticRegression()
row-count check on the leak 2 data: 4 rows counted against 6
